# Tutorial 3: Bootstrap Current, Redl, And Optimization

This notebook shows where bootstrap-current validation and optimization enter the `sfincs_jax` workflow. It is written as a guide: the Redl and VMEC-JAX examples need external equilibrium data, so the heavy commands are shown explicitly and can be run when those data are available.

## Bootstrap Current Observable

For RHSMode=1 profile-current calculations, `sfincs_jax` reports the flux-surface-averaged current diagnostic `FSABjHatOverRootFSAB2`. In SI units the common comparison quantity is

$$\frac{\langle \mathbf{J}\cdot\mathbf{B}\rangle}{\sqrt{\langle B^2\rangle}},$$

which can be compared with SFINCS Fortran v3 frozen outputs or the Redl bootstrap-current fit when the same profiles, normalization, and radial coordinate are used.

## Fast User-Facing Redl Comparison

The script below makes one plot: `sfincs_jax` versus Redl. It does not require a local SFINCS Fortran executable. It does require the QS-paper Zenodo decks and an importable `vmec_jax` checkout for Redl algebra.

In [ ]:
cmd = """
python examples/vmec_jax_finite_beta/compare_qs_paper_sfincs_jax_redl.py \
  --case QA \
  --quick \
  --jax-vs-redl \
  --solve-method auto \
  --stem tutorial_qa_jax_redl_quick
""".strip()
print(cmd)

For a quasi-helical benchmark, change `--case QA` to `--case QH`. For a production validation figure, increase the grid, add `--with-errorbars`, and require same-resolution Fortran overlays only if the archived or rerun Fortran files match the `sfincs_jax` grid.

In [ ]:
production_cmd = """
python examples/vmec_jax_finite_beta/compare_qs_paper_sfincs_jax_redl.py \
  --case QA \
  --s-values 0.1,0.15,0.25,0.3,0.45,0.5,0.6,0.7,0.75,0.85,0.9 \
  --ntheta 13 --nzeta 13 --nxi 21 --nx 5 \
  --with-errorbars \
  --real-ntheta 15 --real-nzeta 15 \
  --velocity-nxi 25 --velocity-nx 6 \
  --solver-tolerance 1e-6 \
  --solve-method auto \
  --jax-vs-redl
""".strip()
print(production_cmd)

## Optimization Workflow

Optimization should normally be two-tiered. Use a fast differentiable proxy objective in the optimizer, then promote accepted candidates to full kinetic `sfincs_jax` scans before making physics claims.

In [ ]:
!python examples/optimization/qa_nfp2_sfincs_jax_objectives.py --objective balanced --steps 20 --out-dir tutorial_output --stem tutorial_qa_proxy

The proxy output shows the trend in bootstrap proxy, electron-root drive, particle/heat/impurity flux proxies, and QA regularization. The promotion scripts in `examples/optimization/` then launch actual `sfincs_jax` scan-er runs, CPU/GPU comparisons, and optional SFINCS Fortran v3 overlays.

In [ ]:
from pathlib import Path

sorted(Path("tutorial_output").glob("tutorial_qa_proxy.*"))

## Validation Rules

- Keep `sfincs_jax`, Redl, and any Fortran overlay on the same profiles and radial coordinate.
- Use error bars from real-space and velocity-space refinements for bootstrap-current plots.
- Treat proxy optimization as candidate generation, not as a kinetic validation.
- Use `docs/optimization.rst`, `docs/validation_matrix.rst`, and `docs/testing.rst` for the publication-grade gates.